In [5]:
# Task 3.1: Features, split and baseline
# Step 1: Load the cleaned dataset
import pandas as pd

# Load the cleaned appointments dataset
df = pd.read_csv("appointments_clean.csv")

# Display dataset information
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

# Display first 5 rows
df.head()


Dataset shape: (990, 16)

Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64.0,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [2]:
# Step 2: Select features and target
# Select the required input features
X = df[[
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type"
]].copy()

# Convert reminder_sent into numeric values
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Apply one-hot encoding to categorical columns
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# Select the target column
y = df["no_show"]

# Display the results
print("Feature shape:", X.shape)
print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget shape:", y.shape)


Feature shape: (990, 11)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Target shape: (990,)


In [3]:
# Step 3: Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Display dataset sizes
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

# Count no-shows in the test set
print("No-shows in test set:", y_test.sum())


Training features: (792, 11)
Testing features: (198, 11)

Training records: 792
Testing records: 198
No-shows in test set: 49


In [4]:
# Step 4: Calculate baseline accuracy
from sklearn.metrics import accuracy_score

# Predict Attended (0) for every test appointment
baseline_predictions = [0] * len(y_test)

# Calculate baseline accuracy
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Baseline Accuracy:", round(baseline_accuracy, 4))
print("Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))


Baseline Accuracy: 0.7525
Baseline Accuracy (%): 75.25


In [8]:
# Task 3.2 – Logistic Regression
# Step 1: Scale the features

from sklearn.preprocessing import StandardScaler

# Create a scaler
scaler = StandardScaler()

# Fit scaler only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Testing data scaled:", X_test_scaled.shape)
print("Feature scaling completed successfully!")



Training data scaled: (792, 11)
Testing data scaled: (198, 11)
Feature scaling completed successfully!


In [9]:
# Step 2: Train Logistic Regression model

from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)

# Train the model using training data
lr_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")


Logistic Regression model trained successfully!


In [10]:
# Step 3: Model Evaluation

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predict results on test data
y_pred = lr_model.predict(X_test_scaled)

# Calculate prediction probabilities
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Calculate evaluation metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))


Accuracy: 0.7879
Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889


In [11]:
# Step 4: Display Logistic Regression coefficients

# Display model coefficients for all features
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_model.coef_[0]
})

print(coefficients.to_string(index=False))


                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658


In [12]:
# Step 5: Calculate reminder odds ratio
import numpy as np

# Get the reminder_sent feature index
reminder_index = X.columns.get_loc("reminder_sent")

# Get its coefficient
reminder_coefficient = lr_model.coef_[0][reminder_index]

# Odds ratio per 1 standard deviation
scaled_odds_ratio = np.exp(reminder_coefficient)

# Odds ratio for changing reminder from No (0) to Yes (1)
real_odds_ratio = np.exp(
    reminder_coefficient / scaler.scale_[reminder_index]
)

print("Reminder coefficient:", round(reminder_coefficient, 4))
print("Odds ratio (per 1 SD):", round(scaled_odds_ratio, 4))
print("Real No-to-Yes odds ratio:", round(real_odds_ratio, 4))
print("Percentage change in odds:",
      round((real_odds_ratio - 1) * 100, 2), "%")


Reminder coefficient: -0.5807
Odds ratio (per 1 SD): 0.5595
Real No-to-Yes odds ratio: 0.3094
Percentage change in odds: -69.06 %


In [13]:
# Task 3.3 – Decision Tree Models
# Step 1: Train both models

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Full Decision Tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

# Decision Tree with maximum depth of 3
dt_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
dt_depth3.fit(X_train, y_train)

# Calculate training and testing accuracy
print("Full Decision Tree")
print("Training Accuracy:",
      round(accuracy_score(y_train, dt_full.predict(X_train)), 4))
print("Testing Accuracy:",
      round(accuracy_score(y_test, dt_full.predict(X_test)), 4))

print("\nDecision Tree (Max Depth = 3)")
print("Training Accuracy:",
      round(accuracy_score(y_train, dt_depth3.predict(X_train)), 4))
print("Testing Accuracy:",
      round(accuracy_score(y_test, dt_depth3.predict(X_test)), 4))


Full Decision Tree
Training Accuracy: 1.0
Testing Accuracy: 0.6717

Decision Tree (Max Depth = 3)
Training Accuracy: 0.822
Testing Accuracy: 0.7778


In [14]:
# Task 3.4 – Random Forest
# Step 1: Train the Random Forest model

from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")


Random Forest model trained successfully!


In [15]:
# Step 2: Evaluate Random Forest

from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Predict test results
rf_pred = rf_model.predict(X_test)

# Predict no-show probabilities
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
print("Random Forest Accuracy:",
      round(accuracy_score(y_test, rf_pred), 4))

print("Random Forest Recall:",
      round(recall_score(y_test, rf_pred), 4))

print("Random Forest ROC-AUC:",
      round(roc_auc_score(y_test, rf_prob), 4))


Random Forest Accuracy: 0.7475
Random Forest Recall: 0.3061
Random Forest ROC-AUC: 0.7057


In [16]:
# Step 3: Find the top 3 important features

# Get feature importance values
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

# Sort features by importance
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

# Display top 3 features
print("Top 3 Important Features:")
print(feature_importance.head(3).to_string(index=False))


Top 3 Important Features:
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739


In [17]:
# Step 4: Compare the three models

from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Predictions from Depth-3 Decision Tree
dt3_pred = dt_depth3.predict(X_test)
dt3_prob = dt_depth3.predict_proba(X_test)[:, 1]

# Create model comparison table
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, dt3_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, dt3_pred),
        recall_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, dt3_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

print(comparison.round(4).to_string(index=False))


                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (Depth 3)    0.7778  0.2245   0.6997
          Random Forest    0.7475  0.3061   0.7057


In [18]:
# Task 3.5 – Model Tuning and Final Selection
# Step 1: Hyperparameter tuning using GridSearchCV

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameter combinations
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

# Train using training data only
grid_search.fit(X_train, y_train)

# Display best results
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation ROC-AUC:",
      round(grid_search.best_score_, 4))


Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation ROC-AUC: 0.7673


In [19]:
# Step 2: Compare all four models using 5-fold cross-validation

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Define the four models
models = {
    "Logistic Regression": lr_pipe,

    "Decision Tree (Depth 3)": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Tuned Random Forest": grid_search.best_estimator_
}

# Calculate 5-fold cross-validation ROC-AUC
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    )

    cv_results.append({
        "Model": name,
        "Mean CV ROC-AUC": scores.mean()
    })

# Display comparison
cv_comparison = pd.DataFrame(cv_results)

print(cv_comparison.round(4).to_string(index=False))


                  Model  Mean CV ROC-AUC
    Logistic Regression           0.7737
Decision Tree (Depth 3)           0.6784
          Random Forest           0.7466
    Tuned Random Forest           0.7673


In [20]:
# Step 3: Select the best classification threshold

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Generate out-of-fold probabilities using training data
oof_prob = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare two thresholds
for threshold in [0.5, 0.35]:

    predictions = (oof_prob >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Precision:",
          round(precision_score(y_train, predictions), 4))
    print("Recall:",
          round(recall_score(y_train, predictions), 4))
    print("F1 Score:",
          round(f1_score(y_train, predictions), 4))
    print("Number of appointments flagged:",
          predictions.sum())



Threshold: 0.5
Precision: 0.5753
Recall: 0.2485
F1 Score: 0.3471
Number of appointments flagged: 73

Threshold: 0.35
Precision: 0.513
Recall: 0.4675
F1 Score: 0.4892
Number of appointments flagged: 154


In [21]:
# Step 4: Final model evaluation on test data

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Train the selected model on training data
lr_pipe.fit(X_train, y_train)

# Predict probabilities on test data
final_prob = lr_pipe.predict_proba(X_test)[:, 1]

# Apply selected threshold
final_pred = (final_prob >= 0.35).astype(int)

# Evaluate final model
print("Final Model: Logistic Regression")
print("Selected Threshold: 0.35")

print("\nAccuracy:",
      round(accuracy_score(y_test, final_pred), 4))

print("Confusion Matrix:\n",
      confusion_matrix(y_test, final_pred))

print("Precision:",
      round(precision_score(y_test, final_pred), 4))

print("Recall:",
      round(recall_score(y_test, final_pred), 4))

print("F1 Score:",
      round(f1_score(y_test, final_pred), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, final_prob), 4))

print("Appointments Flagged:", final_pred.sum())


Final Model: Logistic Regression
Selected Threshold: 0.35

Accuracy: 0.7879
Confusion Matrix:
 [[128  21]
 [ 21  28]]
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7889
Appointments Flagged: 49


In [22]:
# Final Step: Save test_risk_scores.csv

# Create final test risk scores
test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": final_prob,
    "flag": final_pred
})

# Save the CSV file
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

# Verify the saved data
print("File saved successfully!")
print("File shape:", test_risk_scores.shape)
print("\nFirst 5 rows:")
print(test_risk_scores.head())

print("\nTotal flagged appointments:", test_risk_scores["flag"].sum())


File saved successfully!
File shape: (198, 4)

First 5 rows:
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0

Total flagged appointments: 49
